# Pre-model EDA — Customer Attributes

**Audience:** anyone building or reviewing the credit-decisioning model with the 5 customer-level attributes.

**Scope:** strictly pre-model exploration of the 5 attributes (`credit_score`, `annual_income`, `account_tenure_months`, `n_products`, `prev_delinquency_count`) and their relationship to the latent ground-truth response parameters (`true_p_default`, `true_p_accept_cli`, `true_delta_spend_if_accept`). No model is trained here — that lives in the training_flow pipeline. Post-model evaluation (calibration, Gini/KS, vintage on predictions, error analysis) lives in `ml_evaluation.ipynb`.

**Data source:** synthetic cohort generated in-process via `transactions.customer.generate_cohort(size=10000, seed=42)`. No live cluster needed; this notebook is fully reproducible from a clean checkout.

**What this notebook proves before model training:**
1. Distributions per segment look reasonable (no implausible values, no collapsed segments)
2. The 5 attributes carry signal — they correlate with `true_p_default` in the expected direction
3. **Monotonicity preconditions hold** for the attributes that should drive monotonic-constrained model training (Phase B Item S5): credit_score↓ → PD↑, tenure↓ → PD↑, income↓ → PD↑, prev_delinquency↑ → PD↑
4. Bucket-by-attribute default rates are non-trivial — Phase B Item S6 (WoE binning + IV ranking) has signal to work with

**What this notebook deliberately does NOT do:**
- Train a model (that's in `training_flow/`)
- Compute calibration / Gini / KS / Lorenz — those are post-model, in `ml_evaluation.ipynb`
- Use real Kafka-flowed data — synthetic cohort is intentional for reproducibility

## §1 Setup

Standard imports and an in-process cohort of 10k customers seeded for reproducibility.

In [ ]:
import sys
from pathlib import Path

# Make the transactions package importable from the notebook regardless of CWD
_repo = Path.cwd()
while _repo.name and not (_repo / 'services').exists():
    _repo = _repo.parent
sys.path.insert(0, str(_repo / 'services' / 'transactions' / 'src'))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from transactions.customer import generate_cohort
from transactions.segments import SEGMENTS

sns.set_theme(style='whitegrid', context='notebook')

ATTRS = [
    'credit_score',
    'annual_income',
    'account_tenure_months',
    'n_products',
    'prev_delinquency_count',
]
TARGETS = [
    'true_p_default',
    'true_p_accept_cli',
    'true_delta_spend_if_accept',
]

MASTER_SEED = 42
COHORT_SIZE = 10000

cohort = generate_cohort(size=COHORT_SIZE, seed=MASTER_SEED)

# Build a tidy DataFrame for analysis
rows = []
for c in cohort:
    rows.append(
        {
            'customer_id': c.customer_id,
            'segment_id': int(c.segment_id),
            'segment_name': SEGMENTS[int(c.segment_id)].name,
            **{a: getattr(c, a) for a in ATTRS},
            **{t: getattr(c, t) for t in TARGETS},
        }
    )
df = pd.DataFrame(rows)
print(f'Cohort size: {len(df)}')
print(f'Segments: {df["segment_name"].value_counts().sort_index()}')
df.head(3)

## §2 Per-attribute distributions, hued by segment

What we expect to see: smooth, well-separated per-segment distributions for `credit_score`, `annual_income`, `account_tenure_months`; Poisson-like discrete distributions for `n_products` and `prev_delinquency_count`. Low-risk segments (0, 1) shifted to favorable values; high-risk segments (4, 5) shifted to unfavorable.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for ax, attr in zip(axes, ATTRS, strict=False):
    for sid in sorted(df['segment_id'].unique()):
        sub = df[df['segment_id'] == sid][attr]
        ax.hist(sub, bins=30, alpha=0.45, label=f'seg{sid}', density=True)
    ax.set_title(attr)
    ax.set_xlabel(attr)
    ax.set_ylabel('density')
    ax.legend(fontsize=7, loc='best')
axes[-1].axis('off')
plt.tight_layout()
plt.show()

## §3 Correlation structure

Pearson correlation between the 5 attributes and the 3 latent targets. Expected directions per the DGP design:

| Pair | Expected sign | Why |
|------|---------------|-----|
| credit_score × true_p_default | **negative** | Higher score → lower default risk (textbook) |
| annual_income × true_p_default | negative (weak) | Capacity-to-pay effect |
| account_tenure_months × true_p_default | negative (weak) | Relationship effect |
| prev_delinquency_count × true_p_default | **positive** | Past behavior predicts future risk |
| annual_income × true_delta_spend_if_accept | **positive (strong)** | Higher income → more capacity for incremental spend |
| credit_score × true_p_accept_cli | positive (weak) | Higher-score customers qualify for more offers |

In [ ]:
corr_cols = ATTRS + TARGETS
corr = df[corr_cols].corr(method='pearson')

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1,
    vmax=1,
    cbar_kws={'label': 'Pearson r'},
    ax=ax,
)
ax.set_title('Pearson correlation: customer attrs × latent response targets')
plt.tight_layout()
plt.show()

print('Key target correlations (pooled across segments):')
for attr in ATTRS:
    print(f'  {attr:30}  vs true_p_default = {corr.loc[attr, "true_p_default"]:+.4f}')

## §4 Default-rate by attribute bucket — monotonicity precondition (Phase B S5 prep)

For each attribute, bin into deciles and compute the **mean true_p_default per bucket**. This is the precondition check for monotonic-constrained model training (Phase B Item S5): if the raw relationship in the DGP is NOT monotonic, enforcing monotonicity at training time would harm fit. We expect:
- credit_score: monotonically **decreasing** PD as score increases
- annual_income: monotonically **decreasing** PD as income increases
- account_tenure_months: monotonically **decreasing** PD as tenure grows
- n_products: weakly decreasing (relationship breadth)
- prev_delinquency_count: monotonically **increasing** PD with delinquency count

In [ ]:
def bucket_mean_pd(df: pd.DataFrame, attr: str, n_bins: int = 10) -> pd.DataFrame:
    """Equal-frequency binning; returns bucket → mean(true_p_default)."""
    df_local = df.copy()
    if df_local[attr].nunique() <= n_bins:
        # Discrete attribute (n_products, prev_delinquency_count) — use value as bucket
        df_local['bucket'] = df_local[attr]
    else:
        df_local['bucket'] = pd.qcut(
            df_local[attr], q=n_bins, duplicates='drop', labels=False
        )
    g = (
        df_local.groupby('bucket')
        .agg(
            n=(attr, 'size'),
            attr_mean=(attr, 'mean'),
            pd_mean=('true_p_default', 'mean'),
            pd_std=('true_p_default', 'std'),
        )
        .reset_index()
    )
    return g


def monotonicity_sign(g: pd.DataFrame) -> str:
    """Spearman rank correlation between attr_mean and pd_mean across buckets."""
    rho = g[['attr_mean', 'pd_mean']].corr(method='spearman').iloc[0, 1]
    if rho < -0.7:
        return f'STRONG NEGATIVE monotonic (rho={rho:+.3f})'
    if rho > 0.7:
        return f'STRONG POSITIVE monotonic (rho={rho:+.3f})'
    if abs(rho) < 0.3:
        return f'NON-MONOTONIC (rho={rho:+.3f}) — do NOT enforce monotonicity'
    return f'WEAK monotonic (rho={rho:+.3f})'


fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
summary = []
for ax, attr in zip(axes, ATTRS, strict=False):
    g = bucket_mean_pd(df, attr)
    ax.bar(range(len(g)), g['pd_mean'], color='steelblue', alpha=0.8)
    ax.set_title(f'{attr}\n{monotonicity_sign(g)}', fontsize=10)
    ax.set_xlabel('decile (or value)')
    ax.set_ylabel('mean true_p_default')
    ax.set_xticks(range(len(g)))
    ax.set_xticklabels(
        [f'{v:.1f}' if isinstance(v, float) else str(v) for v in g['attr_mean']],
        rotation=45,
        fontsize=7,
    )
    summary.append({'attr': attr, 'monotonicity': monotonicity_sign(g)})
axes[-1].axis('off')
plt.tight_layout()
plt.show()

print('\nPhase B S5 monotonicity verdict per attribute:')
for s in summary:
    print(f'  {s["attr"]:30}  {s["monotonicity"]}')

## §5 Cross-attribute interaction: credit_score × annual_income → PD

Two-dimensional bucketed view. A real-credit-model invariant: low score AND low income should produce the highest PD; high score AND high income should produce the lowest. Anything that violates this would indicate the DGP has unintended interactions worth investigating.

In [ ]:
df_iv = df.copy()
df_iv['score_decile'] = pd.qcut(
    df_iv['credit_score'], q=10, duplicates='drop', labels=False
)
df_iv['income_decile'] = pd.qcut(
    df_iv['annual_income'], q=10, duplicates='drop', labels=False
)
pivot = df_iv.pivot_table(
    index='score_decile',
    columns='income_decile',
    values='true_p_default',
    aggfunc='mean',
)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    pivot,
    annot=True,
    fmt='.3f',
    cmap='Reds',
    ax=ax,
    cbar_kws={'label': 'mean true_p_default'},
)
ax.set_title(
    'Mean PD by (credit_score decile × annual_income decile)\nLow=worst score/income; High=best'
)
ax.set_xlabel('annual_income decile (0=low → 9=high)')
ax.set_ylabel('credit_score decile (0=low → 9=high)')
plt.tight_layout()
plt.show()

top_left = pivot.iloc[0, 0]
bottom_right = pivot.iloc[-1, -1]
print(f'Highest-risk bucket (lowest score AND lowest income): mean PD = {top_left:.4f}')
print(
    f'Lowest-risk bucket  (highest score AND highest income): mean PD = {bottom_right:.4f}'
)
print(f'Risk multiplier (corner to corner): {top_left / max(bottom_right, 1e-9):.1f}x')

## §6 Per-segment summary table

Mean and std of every customer attribute + every target, grouped by segment. Quick sanity check that the segment calibration in `segments.py` produced the intended population structure.

In [ ]:
summary = (
    df.groupby(['segment_id', 'segment_name'])[ATTRS + TARGETS]
    .agg(['mean', 'std'])
    .round(3)
)
summary

## §7 Takeaways for downstream phases

**Phase B Item S5 — Monotonic constraints (model-side).** Use the §4 verdict to decide which features get monotonic constraints in the XGBoost variant. Strong-monotonic features (likely credit_score ↓, prev_delinquency ↑) MUST get constraints for regulatory acceptance. Non-monotonic ones must NOT get constraints — that would harm fit and produce miscalibration.

**Phase B Item S6 — WoE / IV scorecard (pre-model, extends THIS notebook with §8).** The §4 bucketed default-rate tables are the raw material for WoE encoding. The Information Value (IV) per attribute will rank predictive power; expect credit_score and prev_delinquency_count to have the highest IV.

**Step 12 — cluster apply / retrain.** When the producer is rebuilt with the 5 fields and the training pipeline retrains on 26 features, the model's feature_cols list will grow from 21 to 26. The §3 correlations predict which new features should show up high in the model's feature importance ranking after training (post-model analysis lives in `ml_evaluation.ipynb` Section 12 update — Step 11b).

**Phase B Item S1 — Calibration (post-model).** With the 5 attributes carrying real signal (per §3 / §4), the trained 26-feature model should be **better calibrated** than the 21-feature model. Calibration plots will quantify this — landing in `ml_evaluation.ipynb` §13.

**Cohort regeneration note.** Adding the 5 attribute draws to `generate_cohort()` shifts the RNG sequence. Pre-existing cached cohorts from before this change will not bit-match. This is expected; we are intentionally regenerating with richer features.

**Cloud-agnostic reminder.** This notebook uses only `transactions.customer.generate_cohort()` — no Kafka, no RisingWave, no MLflow. Runs identically on any cloud (or no cloud).

## §8 Weight of Evidence + Information Value — Phase B Item S6

**Why this matters.** Traditional credit scorecards (the model that actually deploys at JPMC, Capital One, Discover, FICO) are logistic regression on Weight-of-Evidence-encoded features, ranked by Information Value. Even when a bank ships a GBM or neural net in production, the WoE+IV analysis is the first thing model risk reviewers ask for, because it:

1. **Ranks features by predictive power** in a way regulators understand (IV is in their handbook)
2. **Detects non-monotonic relationships** that would break a monotonic-constrained model (Phase B Item S5)
3. **Produces interpretable bin-level event rates** that adverse-action explanations can cite directly (ECOA / Reg B)

**Mathematical setup.** For a binary target $y \in \{0, 1\}$ (default vs no default), and feature $x$ binned into $K$ bins indexed by $b$:

$$ \text{WoE}_b = \ln\left(\frac{\text{dist}_{\text{non-event},b}}{\text{dist}_{\text{event},b}}\right) \qquad \text{IV} = \sum_{b=1}^{K} (\text{dist}_{\text{non-event},b} - \text{dist}_{\text{event},b}) \cdot \text{WoE}_b $$

Siddiqi (2017) IV strength conventions: <0.02 unpredictive, 0.02-0.10 weak, 0.10-0.30 medium, 0.30-0.50 strong, >0.50 suspicious (target leakage warning).

**Binary target from continuous PD.** Our DGP exposes `true_p_default` as a latent probability. Real credit observations are binary default events. To compute IV/WoE meaningfully, we sample one Bernoulli realization per row via `simulate_binary_target`. Seeded — deterministic across runs.

In [ ]:
from training_flow.woe_scorecard import (
    compute_woe,
    iv_strength_label,
    rank_features_by_iv,
    simulate_binary_target,
)

# Simulate binary default events from the latent true_p_default
df['default_event'] = simulate_binary_target(df['true_p_default'], seed=MASTER_SEED)
print(f'Cohort: {len(df)} customers')
print(f'Realized default rate: {df["default_event"].mean():.4f}')
print(f'Mean true_p_default:   {df["true_p_default"].mean():.4f}')
print('(Should match within sampling noise — verifies the simulator)')

### §8.1 IV ranking — which attributes carry the most predictive power?

Expected ranking based on the DGP design + §4 monotonicity findings:
1. `credit_score` — should rank highest (textbook strongest PD predictor; DGP encodes it directly)
2. `prev_delinquency_count` — strong positive monotonic relationship with PD
3. `account_tenure_months` — moderate relationship
4. `annual_income` — weak-to-medium (correlated with credit_score, partial-r effect)
5. `n_products` — weakest (smaller causal coefficient in the DGP)

In [ ]:
iv_ranking = rank_features_by_iv(
    df=df,
    target=df['default_event'],
    feature_cols=ATTRS,
    n_bins=10,
)
iv_ranking

### §8.2 Per-attribute WoE bins — interpretable risk patterns

For each attribute, show the WoE per bin alongside the realized event rate.
Monotonic WoE patterns directly translate to a regulator-friendly scorecard:
each bin contributes a fixed point increment/decrement to the final score.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
for ax, attr in zip(axes, ATTRS, strict=False):
    result = compute_woe(df[attr], df['default_event'], n_bins=10, name=attr)
    bin_labels = [
        f'[{b.lo:.1f}, {b.hi:.1f}]' if b.lo != b.hi else f'{b.lo:.0f}'
        for b in result.bins
    ]
    woes = [b.woe for b in result.bins]
    event_rates = [b.event_rate for b in result.bins]

    ax2 = ax.twinx()
    bars = ax.bar(range(len(woes)), woes, alpha=0.6, color='steelblue', label='WoE')
    (line,) = ax2.plot(
        range(len(event_rates)), event_rates, 'o-', color='red', label='event rate'
    )
    ax.axhline(0, color='black', linewidth=0.6, linestyle='--')
    ax.set_xticks(range(len(woes)))
    ax.set_xticklabels(bin_labels, rotation=45, fontsize=7, ha='right')
    ax.set_title(
        f'{attr}\nIV = {result.iv_total:.3f} ({iv_strength_label(result.iv_total)}); monotonic={result.monotonic_in_event_rate}',
        fontsize=10,
    )
    ax.set_ylabel('WoE', color='steelblue')
    ax2.set_ylabel('event rate', color='red')
axes[-1].axis('off')
plt.tight_layout()
plt.show()

### §8.3 WoE-encoded scorecard baseline — how a traditional credit model would look

If we encoded every customer's 5 attributes with WoE and trained logistic regression
on the encoded features, we'd get a classical credit scorecard. Below we show the
encoded values alongside the raw attributes for the first 5 customers — the WoE
columns are what a regulator-acceptable model would actually consume.

In [ ]:
from training_flow.woe_scorecard import apply_woe

# Fit a WoE binning per attribute and encode the column
woe_results = {}
encoded = pd.DataFrame({'customer_id': df['customer_id']})
for attr in ATTRS:
    res = compute_woe(df[attr], df['default_event'], n_bins=10, name=attr)
    woe_results[attr] = res
    encoded[f'{attr}__woe'] = apply_woe(df[attr], res)

# Show side-by-side: raw + WoE for first few rows
side_by_side = df[['customer_id'] + ATTRS + ['default_event']].head(5)
for attr in ATTRS:
    side_by_side[f'{attr}__woe'] = encoded[f'{attr}__woe'].head(5).values
side_by_side

### §8.4 Takeaways

**For Phase B Item S5 (monotonic constraints).** §4's Spearman-rho-based monotonicity check + §8.1's IV-based monotonicity check should AGREE. Where they agree on "strongly monotonic," apply XGBoost `monotone_constraints=+1` (increasing PD) or `-1` (decreasing PD) at training time. Where they disagree, the feature is **NOT safe** to constrain — doing so would harm calibration.

**For Phase B Item S1 (calibration).** The §8.1 IV ranking predicts which features the post-training calibration analysis will find most "miscalibrated" if dropped. High-IV features (likely credit_score, prev_delinquency) are the ones whose contribution to the score must be calibrated carefully.

**For Phase B Item S4 (PD × LGD × EAD decomposition).** This entire §8 analysis is the *PD modeling* leg. LGD and EAD have their own scorecards (separate models in real shops). The pattern here generalizes — you'd rerun §8 for `loss_given_default` and `exposure_at_default` once those columns exist in the DGP. Documented in `docs/scope_expansion_plan.md` Phase B.

**For Step 11b (post-model evaluation, after Step 12.6 retrains).** The IV ranking here gives the *prior expectation* for feature importance in the trained model. If the model's SHAP-derived ranking matches this WoE-derived ranking, that's evidence the model learned the right structure. Big disagreements warrant investigation (overfit to noise, label leakage, or a missed interaction).

**Honesty note on synthetic data.** A real bank's WoE/IV table would show monotonic-relationship breakdowns indicating macroeconomic regime shifts (e.g., during a recession the tenure-vs-PD relationship can flip). Our synthetic DGP has no time-varying regime, so we should expect cleaner monotonicity than reality. Phase D Item A9 (stress testing) will deliberately inject regime shifts to test what the model does when the WoE structure changes mid-deployment.